In [1]:
%load_ext autoreload
%autoreload 2

# Simulated background components, DC4 511 keV binning

Binned with the workflow's `dataio_511.yaml` (`BinnedData`): Em 509–513 keV (1 bin), Phi 60 × 3°, PsiChi NSIDE 16 RING (Galactic and local), Time 2217 bins.

| section | content |
|---|---|
| 2 | part 01 (cosmic photons) |
| 3 | parts 02–04; part 05 (SAA) binned only to remove it from the observation |
| 4 | NF samples of part 01 (validation only) |

## 1. Setup

### 1.1 Imports

In [2]:
import sys
from pathlib import Path

import numpy as np
import h5py
import astropy.units as u
from astropy.io import fits
from histpy import Histogram

from astropy.coordinates import SkyCoord
from astropy.time import Time
from scoords import SpacecraftFrame
import healpy as hp
import matplotlib.pyplot as plt

from cosipy.data_io import BinnedData
from cosipy.spacecraftfile import SpacecraftHistory

sys.path.insert(0, str(Path.cwd().parent.parent))   # project root, so `src` is importable
from src.nf_binned_template import galactic_conversion_matrix   # local -> Galactic rotation (Section 4.2)
from src.plot_style import set_global_style
set_global_style()

%matplotlib inline


WARNING FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.



### 1.2 Configuration

In [3]:
# DC4 511 keV analysis files, used in place (read-only) from the workflow repository
WORKFLOW_DATA = Path("/Users/maura/Documents/COSI/data-challenge-4/workflow/data")
BINNING_YAML = WORKFLOW_DATA / "config/dataio_511.yaml"
RESPONSE_LOCAL = WORKFLOW_DATA / ("inputs/Response511.o4.e509_513.s20881894470591.m2555"
                                  ".filtered.nonsparse.binnedimaging.imagingresponse.h5")
RESPONSE_GALACTIC = WORKFLOW_DATA / "inputs/extended_source_response_511_merged.h5"
TOTAL_BKG_BINNED = WORKFLOW_DATA / "outputs/simulated_total_background_511_binned.hdf5"
cfg = BinnedData(input_yaml=str(BINNING_YAML))   # binning settings and time window (tmin, tmax) of the yaml

# Component 01 (cosmic photons), compressed file (decompressed in memory, ~11 GB)
SIM_PATH = Path("../../data/background_components/bckgn_components_gz/"
                "CosmicPhotons_3months_unbinned_data_filtered_with_SAAcut.fits.gz")

# Parts 02-04: simulated files grouped as the NF models (as requested from Pascal; part 05 = SAA, excluded)
SIM_DIR = Path("../../data/background_components")      # binned outputs
GZ_DIR = SIM_DIR / "bckgn_components_gz"                # compressed event files (inputs)
DET = "_WithDetCstunbinned_data_filtered_with_SAAcut.fits.gz"
M3 = "_3months_unbinned_data_filtered_with_SAAcut.fits.gz"
SIM_GROUPS = {
    2: ["AlbedoPhotons" + DET],
    3: ["PrimaryPositrons" + DET, "SecondaryPositrons" + M3],
    4: ["AlbedoNeutrons" + DET, "PrimaryAlphas" + DET, "PrimaryElectrons" + DET,
        "PrimaryProtons" + DET, "SecondaryElectrons" + M3, "SecondaryProtons" + DET],
}
MODEL_COUNTS = {2: 27_081_364, 3: 6_176_355, 4: 25_705_670}   # model "counts", notebook 13 Section 2

# Outputs: new files, histpy refuses to overwrite
FRAMES = ("galactic", "local")
OUT_PATHS = {frame: Path(f"../../data/background_components/part-01_simulated_511keV_DC4binning_{frame}.hdf5")
             for frame in FRAMES}                                     # part 01
SIM_OUT_PATHS = {(part, frame): SIM_DIR / f"part-{part:02d}_simulated_511keV_DC4binning_{frame}.hdf5"
                 for part in SIM_GROUPS for frame in FRAMES}          # parts 02-04

# Energy pre-selection only: wider than the Em axis [509, 513] keV, so the histogram edges decide
E_PRE_LO, E_PRE_HI = 508.0, 514.0
CHUNK = 1_000_000
COLUMNS = ("TimeTags", "Energies", "Phi", "Chi local", "Psi local", "Chi galactic", "Psi galactic")

### 1.3 Helpers

In [4]:
def write_new(hist, path):
    # Write a histogram only if the file does not exist yet (never overwrite).
    if path.exists():
        print("exists, not overwritten:", path.resolve())
    else:
        hist.write(path)
        print("written:", path.resolve())


def read_window(path):
    # Events of one file in the pre-selection window [E_PRE_LO, E_PRE_HI), read in chunks (as Section 2.1)
    with fits.open(path, memmap=True) as hdul:
        ev = hdul[1].data
        sel = {k: [] for k in COLUMNS}
        for a in range(0, len(ev), CHUNK):
            energy = ev["Energies"][a:a + CHUNK]
            keep = (energy >= E_PRE_LO) & (energy < E_PRE_HI)
            for k in COLUMNS:
                sel[k].append(np.array(ev[k][a:a + CHUNK][keep], dtype=np.float64))   # copy, native byte order
    return {k: np.concatenate(v) for k, v in sel.items()}

## 2. Simulation, part 01 (cosmic photons)

### 2.1 Read and keep 508–514 keV

Compressed file, decompressed in memory (~11 GB of RAM), then read in chunks (a few minutes).

In [20]:
with fits.open(SIM_PATH, memmap=True) as hdul:
    ev = hdul[1].data
    n_file = len(ev)
    sel = {k: [] for k in COLUMNS}
    for a in range(0, n_file, CHUNK):
        energy = ev["Energies"][a:a + CHUNK]
        keep = (energy >= E_PRE_LO) & (energy < E_PRE_HI)
        for k in COLUMNS:
            sel[k].append(np.array(ev[k][a:a + CHUNK][keep], dtype=np.float64))   # copy, native byte order
events = {k: np.concatenate(v) for k, v in sel.items()}

print(f"{n_file:,} events in file, {len(events['Energies']):,} in {E_PRE_LO:.0f}-{E_PRE_HI:.0f} keV")
for k, v in events.items():
    print(f"  {k:13s} [{v.min():.6f}, {v.max():.6f}]")

97,554,902 events in file, 242,146 in 508-514 keV
  TimeTags      [1835487353.202117, 1843466698.129811]
  Energies      [508.000000, 513.999800]
  Phi           [0.192847, 3.136417]
  Chi local     [0.000000, 6.282732]
  Psi local     [0.000000, 3.141593]
  Chi galactic  [0.000720, 359.999115]
  Psi galactic  [-89.849144, 89.938694]


### 2.2 Time coverage

In [21]:
t, E = events["TimeTags"], events["Energies"]
print(f"yaml: tmin = {cfg.tmin}, tmax = {cfg.tmax}, time_bins = {cfg.time_bins} s")
print(f"component 01 (pre-selected): t in [{t.min():.6f}, {t.max():.6f}]")
print(f"pre-selected events outside [tmin, tmax]: {np.sum((t < cfg.tmin) | (t > cfg.tmax)):,}")

n_in_range = np.sum((E >= 509.0) & (E < 513.0) & (t >= cfg.tmin) & (t < cfg.tmax))
print(f"events in 509-513 keV and [tmin, tmax): {n_in_range:,}")

yaml: tmin = 1835487300.036206, tmax = 1843467254.887687, time_bins = 3600.0 s
component 01 (pre-selected): t in [1835487353.202117, 1843466698.129811]
pre-selected events outside [tmin, tmax]: 0
events in 509-513 keV and [tmin, tmax): 162,652


### 2.3 Bin (Galactic and local)

Both totals must equal `n_in_range`.

In [5]:
binned = {}
for frame in FRAMES:
    bd = BinnedData(input_yaml=str(BINNING_YAML))
    bd.cosi_dataset = events                   # pre-selected events instead of re-reading the file
    bd.get_binned_data(psichi_binning=frame)   # defaults as in the workflow: sparse, no overflow tracking
    binned[frame] = bd.binned_data
    print(f"{frame:9s} axes {list(binned[frame].axes.labels)}, shape {binned[frame].contents.shape}, "
          f"total {float(binned[frame].contents.sum()):,.0f}")

NameError: name 'events' is not defined

### 2.4 Check the axes against the responses and the workflow total background

All must be `True`.

In [23]:
def h5_axis(path, name):
    # Values and attributes of one stored histpy axis; the response matrix itself is not read.
    with h5py.File(path, "r") as f:
        return f[name][:], dict(f[name].attrs)


same = lambda a, b: np.allclose(a, b, rtol=0, atol=1e-6)

for frame, (path, group) in {"local": (RESPONSE_LOCAL, "/DRM/AXES"),
                             "galactic": (RESPONSE_GALACTIC, "/hist/axes")}.items():
    h = binned[frame]
    em, _ = h5_axis(path, f"{group}/2")
    phi, _ = h5_axis(path, f"{group}/3")
    _, psichi = h5_axis(path, f"{group}/4")
    ax = h.axes["PsiChi"]
    print(f"--- {frame} histogram vs {path.name}")
    print("  Em edges equal :", same(h.axes["Em"].edges.to_value(u.keV), em))
    print("  Phi edges equal:", same(h.axes["Phi"].edges.to_value(u.deg), phi))
    print(f"  PsiChi binned  : nside={ax.nside}, scheme={ax.scheme}, coordsys={ax.coordsys}")
    print(f"  PsiChi response: nside={psichi['nside']}, scheme={psichi['scheme']}, coordsys={psichi['coordsys']}")

total_bkg = Histogram.open(TOTAL_BKG_BINNED)
print(f"--- galactic histogram vs {TOTAL_BKG_BINNED.name}")
for label in ("Time", "Em", "Phi"):
    print(f"  {label} edges equal:", same(binned["galactic"].axes[label].edges.value,
                                          total_bkg.axes[label].edges.value))
ax = total_bkg.axes["PsiChi"]
print(f"  PsiChi total bkg: nside={ax.nside}, scheme={ax.scheme}, coordsys={ax.coordsys}")

n01, ntot = float(binned["galactic"].contents.sum()), float(total_bkg.contents.sum())
print(f"component 01 / total background in 509-513 keV: {n01:,.0f} / {ntot:,.0f} = {n01 / ntot:.4f}")

--- local histogram vs Response511.o4.e509_513.s20881894470591.m2555.filtered.nonsparse.binnedimaging.imagingresponse.h5
  Em edges equal : True
  Phi edges equal: True
  PsiChi binned  : nside=16, scheme=RING, coordsys=<SpacecraftFrame Frame (attitude=None, obstime=None, location=None)>
  PsiChi response: nside=16, scheme=RING, coordsys=spacecraftframe
--- galactic histogram vs extended_source_response_511_merged.h5
  Em edges equal : True
  Phi edges equal: True
  PsiChi binned  : nside=16, scheme=RING, coordsys=<Galactic Frame>
  PsiChi response: nside=16, scheme=RING, coordsys=galactic
--- galactic histogram vs simulated_total_background_511_binned.hdf5
  Time edges equal: True
  Em edges equal: True
  Phi edges equal: True
  PsiChi total bkg: nside=16, scheme=RING, coordsys=<Galactic Frame>
component 01 / total background in 509-513 keV: 162,652 / 1,374,985 = 0.1183


### 2.5 Write

In [24]:
for frame in FRAMES:
    write_new(binned[frame], OUT_PATHS[frame])

exists, not overwritten: /Users/maura/Documents/COSI/cosi_background/cosipy-background-nf/data/background_components/CosmicPhotons_511keV_DC4binning_galactic.hdf5
exists, not overwritten: /Users/maura/Documents/COSI/cosi_background/cosipy-background-nf/data/background_components/CosmicPhotons_511keV_DC4binning_local.hdf5


## 3. Simulation, parts 02–04

Files grouped as the NF models (`SIM_GROUPS`). Check: events per group = model counts. Needs only Section 1.

In [6]:
# Check: events in each group's files (header NAXIS2, nothing loaded) = the model's training event count
for part, names in SIM_GROUPS.items():
    rows = 0
    for name in names:
        with fits.open(GZ_DIR / name) as hdul:
            rows += hdul[1].header["NAXIS2"]
    print(f"part {part:02d}: {rows:,} events in {len(names)} file(s) | model counts {MODEL_COUNTS[part]:,} | "
          f"equal: {rows == MODEL_COUNTS[part]}")

part 02: 27,081,364 events in 1 file(s) | model counts 27,081,364 | equal: True
part 03: 6,176,355 events in 2 file(s) | model counts 6,176,355 | equal: True
part 04: 25,705,670 events in 6 file(s) | model counts 25,705,670 | equal: True


### 3.1 Bin and write

Same steps as Section 2 (a few minutes).

In [7]:
sim_binned = {}   # (part, frame) -> binned histogram (Time, Em, Phi, PsiChi)
for part, names in SIM_GROUPS.items():
    per_file = [read_window(GZ_DIR / name) for name in names]
    events_part = {k: np.concatenate([e[k] for e in per_file]) for k in COLUMNS}   # all files of this part
    t, E = events_part["TimeTags"], events_part["Energies"]
    n_window = np.sum((E >= 509.0) & (E < 513.0) & (t >= cfg.tmin) & (t < cfg.tmax))
    print(f"part {part:02d}: {len(E):,} events in {E_PRE_LO:.0f}-{E_PRE_HI:.0f} keV, "
          f"{n_window:,} in 509-513 keV and [tmin, tmax)")

    for frame in FRAMES:
        bd = BinnedData(input_yaml=str(BINNING_YAML))
        bd.cosi_dataset = events_part                 # pre-selected events instead of re-reading the files
        bd.get_binned_data(psichi_binning=frame)      # as Section 2.3
        sim_binned[part, frame] = bd.binned_data
        print(f"  {frame:9s} total {float(bd.binned_data.contents.sum()):,.0f}")
        write_new(bd.binned_data, SIM_OUT_PATHS[part, frame])

fill() discarded one or more values due to out-of-bounds coordinate in a dimension without under/overflow tracking


part 02: 530,783 events in 508-514 keV, 397,961 in 509-513 keV and [tmin, tmax)
  galactic  total 397,961
written: /Users/maura/Documents/COSI/cosi_background/cosipy-background-nf/data/background_components/part-02_simulated_511keV_DC4binning_galactic.hdf5


fill() discarded one or more values due to out-of-bounds coordinate in a dimension without under/overflow tracking


  local     total 397,961
written: /Users/maura/Documents/COSI/cosi_background/cosipy-background-nf/data/background_components/part-02_simulated_511keV_DC4binning_local.hdf5


fill() discarded one or more values due to out-of-bounds coordinate in a dimension without under/overflow tracking


part 03: 649,088 events in 508-514 keV, 500,040 in 509-513 keV and [tmin, tmax)
  galactic  total 500,040
written: /Users/maura/Documents/COSI/cosi_background/cosipy-background-nf/data/background_components/part-03_simulated_511keV_DC4binning_galactic.hdf5


... (2 outputs left out)

fill() discarded one or more values due to out-of-bounds coordinate in a dimension without under/overflow tracking


part 04: 305,980 events in 508-514 keV, 229,630 in 509-513 keV and [tmin, tmax)


fill() discarded one or more values due to out-of-bounds coordinate in a dimension without under/overflow tracking


  galactic  total 229,630
written: /Users/maura/Documents/COSI/cosi_background/cosipy-background-nf/data/background_components/part-04_simulated_511keV_DC4binning_galactic.hdf5
  local     total 229,630
written: /Users/maura/Documents/COSI/cosi_background/cosipy-background-nf/data/background_components/part-04_simulated_511keV_DC4binning_local.hdf5


### 3.2 Part 05 (SAA component): only to remove it from the observation

Activation after the SAA passages (the passages themselves are cut). Not used as a background: the mock data contain exactly these events, so binning them lets notebook 14 subtract them from the observation. Needs only Section 1.

In [ ]:
SAA_FILE = "SAA_3months_unbinned_data_filtered_with_SAAcut_statreduced_akaHEPD01result.fits.gz"
SAA_OUT_PATHS = {frame: SIM_DIR / f"part-05_simulated_511keV_DC4binning_{frame}.hdf5" for frame in FRAMES}

events_saa = read_window(GZ_DIR / SAA_FILE)
t, E = events_saa["TimeTags"], events_saa["Energies"]
print(f"part 05: {len(E):,} events in {E_PRE_LO:.0f}-{E_PRE_HI:.0f} keV, "
      f"{np.sum((E >= 509.0) & (E < 513.0) & (t >= cfg.tmin) & (t < cfg.tmax)):,} in 509-513 keV and [tmin, tmax)")
for frame in FRAMES:
    bd = BinnedData(input_yaml=str(BINNING_YAML))
    bd.cosi_dataset = events_saa
    bd.get_binned_data(psichi_binning=frame)      # as Section 2.3
    print(f"  {frame:9s} total {float(bd.binned_data.contents.sum()):,.0f}")
    write_new(bd.binned_data, SAA_OUT_PATHS[frame])

## 4. NF samples of part 01 (validation only)

Used to validate the NF template in notebooks 09–10; kept for reference. Needs Section 2.

### 4.1 Local frame

No times: binned without a Time axis, on the axes of the local simulation histogram.

In [25]:
NF_SAMPLES_PATH = Path("../../data/NF_background/samples/part-01_full_caffeinated.hdf5")
NF_OUT_PATH = Path("../../data/NF_background/samples/part-01_full_caffeinated_511keV_DC4binning_local.hdf5")
NF_COLUMNS = ("Energies", "Phi", "Chi local", "Psi local")

# Read and pre-select the energy window
with h5py.File(NF_SAMPLES_PATH, "r") as f:
    print("model:", f.attrs["model_file"], "|", f.attrs["description"])
    n_nf = f["Energies"].shape[0]
    sel = {k: [] for k in NF_COLUMNS}
    for a in range(0, n_nf, CHUNK):
        energy = f["Energies"][a:a + CHUNK]
        keep = (energy >= E_PRE_LO) & (energy < E_PRE_HI)
        for k in NF_COLUMNS:
            sel[k].append(np.asarray(f[k][a:a + CHUNK][keep], dtype=np.float64))
nf_events = {k: np.concatenate(v) for k, v in sel.items()}
print(f"{n_nf:,} NF samples, {len(nf_events['Energies']):,} in {E_PRE_LO:.0f}-{E_PRE_HI:.0f} keV")

# Same Em, Phi, PsiChi (local) axes as the simulation; filled as BinnedData does for local binning
sim_local = binned["local"].project("Em", "Phi", "PsiChi")
nf_binned = Histogram(sim_local.axes)
coords = SkyCoord(lon=nf_events["Chi local"], lat=np.pi / 2 - nf_events["Psi local"],
                  unit=u.rad, frame=SpacecraftFrame())
nf_binned.fill(u.Quantity(nf_events["Energies"], u.keV),
               u.Quantity(np.rad2deg(nf_events["Phi"]), u.deg),
               coords)

E_nf = nf_events["Energies"]
print(f"NF samples in 509-513 keV: {np.sum((E_nf >= 509.0) & (E_nf < 513.0)):,}")
print(f"NF binned:  axes {list(nf_binned.axes.labels)}, shape {nf_binned.contents.shape}, "
      f"total {float(nf_binned.contents.sum()):,.0f}")
print(f"simulation (local, summed over time): total {float(sim_local.contents.sum()):,.0f}")

write_new(nf_binned, NF_OUT_PATH)

model: nfbackground_part-01_v3-00.fits | Time-independent NF shape samples; no physical times or rate normalization


fill() discarded one or more values due to out-of-bounds coordinate in a dimension without under/overflow tracking


97,554,902 NF samples, 241,048 in 508-514 keV
NF samples in 509-513 keV: 161,582
NF binned:  axes [np.str_('Em'), np.str_('Phi'), np.str_('PsiChi')], shape (1, 60, 3072), total 161,582
simulation (local, summed over time): total 162,652
written: /Users/maura/Documents/COSI/cosi_background/cosipy-background-nf/data/NF_background/part-01_full_caffeinated_511keV_DC4binning_local.hdf5


### 4.2 Galactic frame

Rotated as a histogram: Galactic counts = (local counts / live time) @ Wᵀ (`galactic_conversion_matrix`).

In [26]:
ORI_PATH = Path("../../data/DC4_orientation.ori")   # same file as DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.ori (same MD5)
NF_OUT_PATH_GAL = Path("../../data/NF_background/samples/part-01_full_caffeinated_511keV_DC4binning_galactic.hdf5")

# Attitude and live time in the binning window [tmin, tmax] of the yaml
history = SpacecraftHistory.open(ORI_PATH).select_interval(Time(cfg.tmin, format="unix"),
                                                           Time(cfg.tmax, format="unix"))
livetime = float(history.livetime.to_value(u.s).sum())

psichi = nf_binned.axes["PsiChi"]
W = galactic_conversion_matrix(history, psichi.nside, nest=(psichi.scheme == "NESTED"))   # [s], (3072, 3072)
print(f"live time in [tmin, tmax]: {livetime:,.1f} s | every row of W sums to it: "
      f"{np.allclose(W.sum(axis=1), livetime)}")

gal_axes = binned["galactic"].project("Em", "Phi", "PsiChi").axes   # Em, Phi, PsiChi (Galactic) of the simulation


def to_galactic(hist_local):
    # Local-frame counts over [tmin, tmax] -> Galactic-frame counts: (counts / live time) @ W^T.
    counts_local = np.asarray(hist_local.to_dense().contents)        # (1, 60, 3072)
    return Histogram(gal_axes, contents=(counts_local / livetime) @ W.T)


nf_galactic = to_galactic(nf_binned)
print(f"NF samples: local total {float(nf_binned.contents.sum()):,.0f} -> "
      f"Galactic total {float(nf_galactic.contents.sum()):,.0f}, shape {nf_galactic.contents.shape}")

write_new(nf_galactic, NF_OUT_PATH_GAL)


WARNING FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.



live time in [tmin, tmax]: 6,579,555.0 s | every row of W sums to it: True
NF samples: local total 161,582 -> Galactic total 161,584, shape (1, 60, 3072)
written: /Users/maura/Documents/COSI/cosi_background/cosipy-background-nf/data/NF_background/part-01_full_caffeinated_511keV_DC4binning_galactic.hdf5


### 4.3 Check of the rotation

Simulation: per-event Galactic map vs local map rotated with W. Pulls ~ N(0, 1) if the rotation is right.

In [27]:
sim_direct = np.asarray(binned["galactic"].project("PsiChi").to_dense().contents)   # per-event Galactic coordinates
sim_rotated = np.asarray(to_galactic(sim_local).project("PsiChi").contents)         # local histogram rotated with W
pull = (sim_direct - sim_rotated) / np.sqrt(np.where(sim_rotated > 0, sim_rotated, np.nan))

print(f"totals: direct {sim_direct.sum():,.0f} | rotated {sim_rotated.sum():,.1f}")
print(f"pulls: mean {np.nanmean(pull):+.3f}, std {np.nanstd(pull):.3f}, max |pull| {np.nanmax(np.abs(pull)):.2f}")

fig = plt.figure(figsize=(18, 4.5))
hp.mollview(sim_direct, fig=fig.number, sub=(1, 3, 1), unit="counts",
            title="simulation, direct (per-event Galactic)")
hp.mollview(sim_rotated, fig=fig.number, sub=(1, 3, 2), unit="counts",
            title="simulation, rotated with W")
hp.mollview(pull, fig=fig.number, sub=(1, 3, 3), min=-4, max=4, cmap="RdBu_r",
            title="pull (direct - rotated) / sqrt(rotated)")
plt.show()

totals: direct 162,652 | rotated 162,644.8
pulls: mean -0.001, std 0.999, max |pull| 3.86



WARNING UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True


WARNING UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True


WARNING UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown

